In [ ]:
import jsonimport pandas as pdimport numpy as npimport matplotlib.pyplot as pltimport seaborn as snsfrom sklearn.feature_extraction.text
import TfidfVectorizerfrom sklearn.decomposition
import TruncatedSVDfrom sklearn.manifold
import TSNEfrom sklearn.cluster
import KMeans, AgglomerativeClustering, DBSCANfrom sklearn.metrics
import homogeneity_completeness_v_measure

In [ ]:
with open(r"C:\Users\yuraz\Downloads\texts\texts.json", "r", encoding="utf-8") as file:
  texts = json.load(file)

In [ ]:
print('тип:', type(texts))
print('всего записей:', len(texts))keys = list(texts.keys())
print('\nпервые 5 ключей:')for k in keys[:5]:    print('   ', repr(k))sample = texts[keys[1]]print('\nтип значений:', type(sample))print('длина текста:', len(sample))print('начало текста:', repr(sample[:150]))

In [ ]:
texts.pop('NaN', None)
df_texts = pd.DataFrame(    {'file_name': list(texts.keys()),     'text': list(texts.values())})
print('текстов после чистки:', len(df_texts))

In [ ]:
meta = pd.read_csv(r"C:\Users\yuraz\Downloads\russian_novels_metadata_char.csv")
meta = meta[['file_name', 'general_genre', 'genre', 'author', 'title']]
df = df_texts.merge(meta, on = 'file_name', how = 'inner')
print('после merge:', len(df))
df = df.dropna(subset = ['general_genre']).reset_index(drop = True)
print('с известным жанром:', len(df))
print('\nраспределение жанров:')
print(df['general_genre'].value_counts())

In [ ]:
top = df['general_genre'].value_counts().head(6).index
df['genre_plot'] = df['general_genre'].where(df['general_genre'].isin(top), 'other')
df['genre_plot'].value_counts()

In [ ]:
pdf = TfidfVectorizer(max_features = 5000, min_df = 2)
pdf_cool = pdf.fit_transform(df['text'])
labels = df['genre_plot'].valuesprint(pdf_cool.shape)

In [ ]:
pdf_svd = TruncatedSVD(n_components=50).fit_transform(pdf_cool)print(pdf_svd.shape)

In [ ]:
ks = range(2, 10)
inertias = []for k in ks:    km = KMeans(n_clusters=k, random_state=42, init='random', n_init='auto').fit(pdf_svd)    inertias.append(km.inertia_)plt.plot(ks, inertias, '-o')plt.show()

In [ ]:
tsne = TSNE(n_components=2, random_state=42, perplexity=30, init='random').fit_transform(pdf_svd)

In [ ]:
kmeans = KMeans(n_clusters=7, random_state=0, n_init="auto")
kmeans.fit(pdf_svd)
preds = kmeans.labels_df['cluster'] = predsplt.scatter(    tsne[:, 0],    tsne[:, 1],    c=preds)
plt.show()

In [ ]:
df_tsne = pd.DataFrame({    'x': tsne[:, 0],    'y': tsne[:, 1],    'genre': labels})
plt.figure(figsize = (10,8))sns.scatterplot(    data = df_tsne,    x = 'x', y = 'y',    hue = 'genre',    palette = sns.color_palette('hls', len(df_tsne['genre'].unique())),    s = 60,    alpha = 0.7)
plt.title('t-SNE проекция')
plt.legend(bbox_to_anchor=(1.05, 1), loc=2)
plt.show()

In [ ]:
def eval_clustering(x, y, algorithm, algorithm_args=None, draw=True, ax=None):
  if algorithm_args is None:
    algorithm_args = {}
    algo = algorithm(**algorithm_args)
    algo.fit(x)
    preds = algo.labels_
    homogeneity, completeness, v_measure = homogeneity_completeness_v_measure(y, preds)
    name = f'h = {round(homogeneity, 4)}, c = {round(completeness, 4)}, v_m = {round(v_measure, 4)}'
    if draw:
      if ax:
    ax.scatter(x[:, 0], x[:, 1], c=preds)
    ax.set_title(name)
      else:
        plt.scatter(x[:, 0], x[:, 1], c=preds)
        plt.title(name)
    else:
      print(f'homogeneity: {homogeneity}\ncompleteness: {completeness}\nv_measure: {v_measure}')
  return algo, preds, homogeneity, completeness, v_measure

In [ ]:
fig, ax = plt.subplots(3, 2, figsize=(14, 20))
fig.suptitle('Homogeneity (h), completeness (c), v_measure (v_m)')
x = tsney = labelsparams = {        'n_clusters': 7,        'random_state': 0,        'n_init': "auto"}
params_1 = {        'eps': 2.0,        'min_samples': 5}
eval_clustering(x, y, KMeans, algorithm_args=params, draw=True, ax=ax[0, 0])ax[0, 1].scatter(x[:, 0], x[:, 1], c = pd.factorize(y)[0])ax[0, 1].set_title('Жанры')
eval_clustering(x, y, AgglomerativeClustering, algorithm_args={'n_clusters': 7}, draw=True, ax=ax[1, 0])ax[1, 1].scatter(x[:, 0], x[:, 1], c = pd.factorize(y)[0])ax[1, 1].set_title('Жанры')
eval_clustering(x, y, DBSCAN, algorithm_args=params_1, draw=True, ax=ax[2, 0])ax[2, 1].scatter(x[:, 0], x[:, 1], c = pd.factorize(y)[0])ax[2, 1].set_title('Жанры')plt.tight_layout()plt.show()